In [ ]:
import neuron as neuron

from dataProcessing import getData, getFilename, calculateLatency, calculateVelocity
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from stimulationProtocols import getCOVIDFullTime
import json
from plot import plotLatency, plotRecoveryCycle
import os
from main_CM import run


neuron.load_mechanisms('./MOD_Tigerholm')

In [ ]:
params_orig = {
    "gPump":        -0.0025,
    "gNav17":        0.24686453257354574,
    "gNav17Parent":  0.13115152763095123,
    "gNav18":        0.37673567973121774,
    "gNav18Parent":  0.2343920300579889,
    "gNav19":        0.00017254238997420438,
    "gKs":           0.008865226128662577,
    "gKf":           0.02709394494148292,
    "gH":            0.014140202887083592,
    "gKdr":          0.008469950837206652,
    "gKna":          0.001398204170298818,
    "vRest":        -55,
}


# gNav17/gNav18 scaling is automatically applied to their Parent counterparts
# run only a few simulations to test out
experiments = [
    {}, # initial
    {"gNav17": -0.40, "gNav18": -0.30, "gH": -0.40},
    {"gNav17": -0.40, "gNav18": -0.40, "gH": -0.20},
    {"gNav17": -0.30, "gNav18": -0.30, "gH": -0.40},
    {"gNav17": -0.30, "gNav18": -0.40, "gH": -0.20},
]

PARENT_COUPLED = {"gNav17": "gNav17Parent", "gNav18": "gNav18Parent"}

protocol = 42

def apply_scales(base, scales):
    """Scales ionic conductances while accounting for the fact that some conductances differ in parent branch"""
    p = base.copy()
    for name, dg in scales.items():
        p[name] = base[name] * (1 + dg)
        if name in PARENT_COUPLED:
            parent = PARENT_COUPLED[name]
            p[parent] = base[parent] * (1 + dg)
    return p



params = [apply_scales(params_orig, exp) for exp in experiments]
print(f"Loading {len(params)} simulations.")


In [ ]:
results = []
for param_dict in params:
    results.append(
        getData(
            prot=protocol,
            filetype="spikes",
            scalingFactor=0.1,
            gPump=param_dict['gPump'],
            gNav17=param_dict['gNav17'],
            gNav17Parent=param_dict['gNav17Parent'],
            gNav18=param_dict['gNav18'],  # this is very confusing because I messed up the naming in run.py
            gNav18Parent=param_dict['gNav18Parent'],
            gNav19=param_dict['gNav19'],
            gKs=param_dict['gKs'],
            gKf=param_dict['gKf'],
            gH=param_dict['gH'],
            gKdr=param_dict['gKdr'],
            gKna=param_dict['gKna'],
            vRest=param_dict['vRest']
        )
    )

In [ ]:
data_stim = getData(prot=protocol, filetype="stim")

In [ ]:
def get_metrics(data_aps, data_stim, Slow025HzStart=1, Slow025HzEnd=90, Fast2HzStart=90, Fast2HzEnd=450,
                Fast2HzPost30S=458):
    initial_velocity = calculateVelocity(data_aps, data_stim)[0]
    latency = calculateLatency(data_aps, data_stim, norm=False)[:, 1]
    latency_points = [latency[Slow025HzStart], latency[Fast2HzStart], latency[Fast2HzEnd], latency[Fast2HzPost30S]]
    Slow025StartToEnd = (latency[Slow025HzEnd] - latency[Slow025HzStart]) / latency[Slow025HzStart]
    Slow025EndToFast2HzEnd = (latency[Fast2HzEnd] - latency[Slow025HzEnd]) / latency[Slow025HzEnd]
    # recovery at 30 s (latency at 30 s after 2 Hz stimulation compared to latency before 0.25 Hz stimulation)
    # can be also negative (but unlikely)
    Fast2HzStartToPost30S = (latency[Fast2HzPost30S] - latency[Slow025HzStart]) / latency[Slow025HzStart]
    TimeTo50Percent = 0  # not implemented yet because it is barely changed by the conductancies

    recovery_50_percent_threshold = (latency[Fast2HzStart] + latency[Fast2HzEnd]) / 2

    for n in np.arange(Fast2HzEnd, SimulationEnd):
        # if the latency at number t-th spike is lower than the 50 % recovery threshold
        if latency[n] < recovery_50_percent_threshold:
            # use the time as the time until 50 % recovery
            TimeTo50Percent = getCOVIDFullTime(n) - getCOVIDFullTime(
                Fast2HzStart)  # can be made more precise with linear interpolation
            break

    return (initial_velocity, Slow025StartToEnd, Slow025EndToFast2HzEnd, Fast2HzStartToPost30S, TimeTo50Percent,
            latency_points)

In [ ]:
Slow025HzStart = 1
Slow025HzEnd = 90
Fast2HzStart = 90  # 90 stimulations at 0.25 Hz initially
Fast2HzEnd = 450  # 360 stimulations at 2 Hz
Fast2HzPost30S = Fast2HzEnd + 8  # 32 s after reducing the stimulation frequency from 2 Hz to 0.25 Hz again

SimulationEnd = data_stim.shape[0] - 1 # ugly but works
points = [Slow025HzStart, Slow025HzEnd, Fast2HzStart, Fast2HzEnd, Fast2HzPost30S, SimulationEnd]
points_name = ["InitialVelocity", "Slow025HzStart", "Slow025HzEnd", "Fast2HzStart", "Fast2HzEnd", "Fast2HzPost30S",
               "SimulationEnd"]
slowing_name = [points_name[0], points_name[1] + "-" + points_name[2], points_name[3] + "-" + points_name[4],
                points_name[5], "TimeTo50Percent"]

In [ ]:
metrics = []
metrics_normalized = []
for result in results:
    metric = get_metrics(result, data_stim)[:-1]
    metrics.append(np.array(metric))

    metric_normalized = []
    for i, m in enumerate(metric):
        metric_normalized.append(m / metrics[0][i])
    metrics_normalized.append(np.array(metric_normalized))

In [ ]:
print("Normalized metrics:")
print(metrics_normalized)

In [ ]:
experiment_factors =  [0.93055556, 0.0625    , 1.26396917, 0.86264442, 1.26081081]

Number 2 and number 4 are the best fits, with the 4th one being the best.

In [ ]:
print(metrics_normalized[3])